# Create INdAM Awards (Istituto Nazionale di Alta Matematica "Francesco Severi")

Creates awards for INdAM's own fellowship programmes co-funded by the EU Marie
Curie COFUND action, from the full fellow lists published on the INdAM-COFUND
site (https://www.altamatematica.it/indam-cofund/):
- **INdAM-COFUND 2008** (FP7): 27 post-doc fellows; 20 have a fellow page with
  type (incoming / outgoing / reintegration), dates, project title, host and abstract.
- **INdAM-COFUND 2012** (FP7, PCOFUND-GA-2012-600198): 30 fellows, by call (1-3) and type.
- **INdAM-DP-COFUND 2015** (H2020): 20 doctoral fellows, by call (1-2).

**77 fellowships, 100% fellow name, 26% title/dates/host (2008 fellow pages), 0%
amount** (per-fellow amounts are not published; fellowship programme). Local run
2026-10-01.

**Scope (what is NOT here, and why):** the national research groups' projects are
separate funders (GNAMPA F4320333369, GNCS F4320334079; GNSAGA/GNFM have their own
rows). INdAM's student scholarships (degree enrolment, master's 'avviamento alla
ricerca', study-abroad months), research 'assegni', the visiting-professor
programme and the 2017 Giovani Ricercatori / 2024 'Progetti di Ricerca INdAM'
calls are published only as per-call ranking PDFs (graduatorie degli idonei) or
announcements where the funded subset is not stated explicitly - left for a
follow-up rather than inferred. The EC's own COFUND grants to INdAM are in
CORDIS under the European Commission; these rows are the individual fellowships
INdAM awarded under them.

**Prerequisites:**
- Run `scripts/local/indam_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/indam/indam_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** no per-fellow code is published; citing works name
the programme ("INdAM-COFUND-2012", "INdAM-DP-COFUND-2015", the EU grant
agreement "PCOFUND-GA-2012-600198"), which identify the programme, not a
fellowship. Synthetic key: `{INDAM-COFUND-2008|INDAM-COFUND-2012|INDAM-DP-COFUND-2015}-{fellow-slug}`.
Collapses onto 0 citation shells.

**Names:** split per runbook §2.4.1 with surname particles kept (von der, Van de,
Della); DP-COFUND 2015 surnames taken from the published rankings, which print
them in capitals; four published names fixed explicitly (see NAME_FIXES).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320311030`
- display_name: Istituto Nazionale di Alta Matematica "Francesco Severi"
- ror_id / doi: from `openalex.funders.funders` (ror 01vx64p53, doi 10.13039/100009112)

**Priority:** `556` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.indam_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/indam/indam_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_fellowships FROM openalex.awards.indam_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.indam_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.indam_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320311030 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320311030;

## Step 2: Create INdAM Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.indam_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320311030  -- INdAM "Francesco Severi"
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        g.*,
        named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', CASE WHEN NULLIF(TRIM(g.host_institution), '') IS NOT NULL THEN named_struct(
                'name', TRIM(g.host_institution),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            ) END
        ) AS lead_struct
    FROM openalex.awards.indam_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- 2008 fellow pages give the project title; otherwise name the fellowship
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT(g.programme, ' ', COALESCE(LOWER(g.fellow_type), 'post-doc'), ' fellowship: ', g.name)) as display_name,
    NULLIF(TRIM(g.abstract), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'fellowship' as funding_type,
    CONCAT_WS(' - ', g.programme, g.call, CASE WHEN g.fellow_type IS NOT NULL THEN CONCAT(g.fellow_type, ' fellowship') END) as funder_scheme,
    'indam_cofund_fellows' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    g.lead_struct as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    ARRAY(g.lead_struct) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM lead g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'indam_cofund_fellows' AND priority = 556;

-- Insert into openalex_awards_raw with priority.
-- Priority 556: direct-from-funder ingest of INdAM's published COFUND fellow lists.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    556 as priority
FROM openalex.awards.indam_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.indam_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every key is one of the synthetic programme forms.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^INDAM-(DP-)?COFUND-(2008|2012|2015)-[a-z0-9-]+$' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^INDAM-(DP-)?COFUND-(2008|2012|2015)-[a-z0-9-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.indam_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.indam_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, COUNT(start_date) as with_dates
FROM openalex.awards.indam_awards
GROUP BY funder_scheme ORDER BY funder_scheme;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.indam_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.indam_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage (amounts not published per fellow: waived for this fellowship ingest).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.indam_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'indam_cofund_fellows'
GROUP BY provenance, priority;